# 01 - First look at MovieLens 32M

Goal: look at the raw files (read-only), no cleaning yet.

In [1]:
import duckdb

con = duckdb.connect()

In [2]:
# First 5 ratings
con.sql("SELECT * FROM '../data/raw/ml-32m/ratings.csv' LIMIT 5")

┌────────┬─────────┬────────┬───────────┐
│ userId │ movieId │ rating │ timestamp │
│ int64  │  int64  │ double │   int64   │
├────────┼─────────┼────────┼───────────┤
│      1 │      17 │    4.0 │ 944249077 │
│      1 │      25 │    1.0 │ 944250228 │
│      1 │      29 │    2.0 │ 943230976 │
│      1 │      30 │    5.0 │ 944249077 │
│      1 │      32 │    5.0 │ 943228858 │
└────────┴─────────┴────────┴───────────┘

In [3]:
# First 5 movies
con.sql("SELECT * FROM '../data/raw/ml-32m/movies.csv' LIMIT 5")

┌─────────┬────────────────────────────────────┬─────────────────────────────────────────────┐
│ movieId │               title                │                   genres                    │
│  int64  │              varchar               │                   varchar                   │
├─────────┼────────────────────────────────────┼─────────────────────────────────────────────┤
│       1 │ Toy Story (1995)                   │ Adventure|Animation|Children|Comedy|Fantasy │
│       2 │ Jumanji (1995)                     │ Adventure|Children|Fantasy                  │
│       3 │ Grumpier Old Men (1995)            │ Comedy|Romance                              │
│       4 │ Waiting to Exhale (1995)           │ Comedy|Drama|Romance                        │
│       5 │ Father of the Bride Part II (1995) │ Comedy                                      │
└─────────┴────────────────────────────────────┴─────────────────────────────────────────────┘

In [ ]:
# number of ratings 
con.sql("SELECT COUNT(*) FROM '../data/raw/ml-32m/ratings.csv' ")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     32000204 │
└──────────────┘

In [9]:
# how many different users ? 
con.sql("SELECT COUNT(DISTINCT userId) FROM '../data/raw/ml-32m/ratings.csv' ")

┌────────────────────────┐
│ count(DISTINCT userId) │
│         int64          │
├────────────────────────┤
│                 200948 │
└────────────────────────┘

In [11]:
# how many distinct movies have been rated (so how many distinct movies in the DB Ratings)
con.sql("SELECT COUNT(DISTINCT movieId) FROM '../data/raw/ml-32m/ratings.csv' ")

┌─────────────────────────┐
│ count(DISTINCT movieId) │
│          int64          │
├─────────────────────────┤
│                   84432 │
└─────────────────────────┘

In [15]:
con.sql("SELECT to_timestamp(MIN(timestamp)), to_timestamp(MAX(timestamp)) FROM '../data/raw/ml-32m/ratings.csv'")

┌────────────────────────────────┬────────────────────────────────┐
│ to_timestamp(min("timestamp")) │ to_timestamp(max("timestamp")) │
│    timestamp with time zone    │    timestamp with time zone    │
├────────────────────────────────┼────────────────────────────────┤
│ 1995-01-09 12:46:44+01         │ 2023-10-13 04:29:07+02         │
└────────────────────────────────┴────────────────────────────────┘

In [ ]:
# scale of ratings
con.sql("SELECT DISTINCT rating FROM '../data/raw/ml-32m/ratings.csv' ORDER BY rating ASC")

┌────────┐
│ rating │
│ double │
├────────┤
│    0.5 │
│    1.0 │
│    1.5 │
│    2.0 │
│    2.5 │
│    3.0 │
│    3.5 │
│    4.0 │
│    4.5 │
│    5.0 │
└────────┘

In [18]:
con.sql("SELECT movieId, title FROM '../data/raw/ml-32m/movies.csv' WHERE title LIKE '%Godfather%' OR title LIKE '%Titanic%' OR title LIKE '%Amelie%'")

┌─────────┬──────────────────────────────────────────────────────────────────────────┐
│ movieId │                                  title                                   │
│  int64  │                                 varchar                                  │
├─────────┼──────────────────────────────────────────────────────────────────────────┤
│     858 │ Godfather, The (1972)                                                    │
│    1221 │ Godfather: Part II, The (1974)                                           │
│    1721 │ Titanic (1997)                                                           │
│    2023 │ Godfather: Part III, The (1990)                                          │
│    2157 │ Chambermaid on the Titanic, The (Femme de chambre du Titanic, La) (1998) │
│    3403 │ Raise the Titanic (1980)                                                 │
│    3404 │ Titanic (1953)                                                           │
│    4864 │ Titanica (1992)                

In [19]:
# Step 7: Titanic (movieId 1721), number of ratings and average rating per year (UTC)
con.sql("SELECT year(to_timestamp(timestamp) AT TIME ZONE 'UTC') AS rating_year, COUNT(*) AS nb_ratings, ROUND(AVG(rating), 2) AS avg_rating FROM '../data/raw/ml-32m/ratings.csv' WHERE movieId = 1721 GROUP BY rating_year ORDER BY rating_year").show(max_rows=40)

┌─────────────┬────────────┬────────────┐
│ rating_year │ nb_ratings │ avg_rating │
│    int64    │   int64    │   double   │
├─────────────┼────────────┼────────────┤
│        1997 │         39 │       4.31 │
│        1998 │       1542 │       4.11 │
│        1999 │       3525 │       3.32 │
│        2000 │       2929 │       3.58 │
│        2001 │       1826 │       3.51 │
│        2002 │       1378 │       3.28 │
│        2003 │       1189 │       3.27 │
│        2004 │       1986 │       3.15 │
│        2005 │       2805 │       3.06 │
│        2006 │       2211 │        3.2 │
│        2007 │       2316 │       3.13 │
│        2008 │       2556 │       3.22 │
│        2009 │       1669 │       3.28 │
│        2010 │       1629 │       3.31 │
│        2011 │       1287 │       3.41 │
│        2012 │       1006 │       3.49 │
│        2013 │        665 │       3.51 │
│        2014 │        528 │        3.4 │
│        2015 │       1878 │       3.44 │
│        2016 │       2881 │      

In [20]:
# Step 8: date and time (UTC) of the very first rating of Titanic (movieId 1721)
con.sql("SELECT to_timestamp(MIN(timestamp)) AT TIME ZONE 'UTC' AS first_rating_utc FROM '../data/raw/ml-32m/ratings.csv' WHERE movieId = 1721")

┌─────────────────────┐
│  first_rating_utc   │
│      timestamp      │
├─────────────────────┤
│ 1997-12-19 19:57:55 │
└─────────────────────┘

In [21]:
# Step 9: total number of ratings per year in the whole dataset (all movies), UTC
con.sql("SELECT year(to_timestamp(timestamp) AT TIME ZONE 'UTC') AS rating_year, COUNT(*) AS nb_ratings FROM '../data/raw/ml-32m/ratings.csv' GROUP BY rating_year ORDER BY rating_year").show(max_rows=40)

┌─────────────┬────────────┐
│ rating_year │ nb_ratings │
│    int64    │   int64    │
├─────────────┼────────────┤
│        1995 │          4 │
│        1996 │    1571368 │
│        1997 │     685388 │
│        1998 │     301691 │
│        1999 │    1174629 │
│        2000 │    1912322 │
│        2001 │    1160098 │
│        2002 │     849762 │
│        2003 │    1011000 │
│        2004 │    1139068 │
│        2005 │    1752573 │
│        2006 │    1141704 │
│        2007 │    1023813 │
│        2008 │    1117071 │
│        2009 │     890696 │
│        2010 │     860073 │
│        2011 │     729301 │
│        2012 │     695493 │
│        2013 │     563301 │
│        2014 │     518537 │
│        2015 │    1743866 │
│        2016 │    1918739 │
│        2017 │    1827953 │
│        2018 │    1391057 │
│        2019 │    1385467 │
│        2020 │    1688159 │
│        2021 │    1239912 │
│        2022 │     913105 │
│        2023 │     794054 │
├─────────────┴────────────┤
│ 29 rows     

In [22]:
# Step 11: for each of the 142 nominees and each rating year (UTC): number of ratings and average rating
con.sql("SELECT n.ceremony_year, n.title, n.is_winner, n.movieId, year(to_timestamp(r.timestamp) AT TIME ZONE 'UTC') AS rating_year, COUNT(*) AS nb_ratings, ROUND(AVG(r.rating), 2) AS avg_rating FROM '../data/processed/best_picture_nominees_1996_2017.csv' n JOIN '../data/raw/ml-32m/ratings.csv' r ON n.movieId = r.movieId GROUP BY n.ceremony_year, n.title, n.is_winner, n.movieId, rating_year ORDER BY n.ceremony_year, n.title, rating_year").show(max_rows=30)


┌───────────────┬───────────────────────┬───────────┬─────────┬─────────────┬────────────┬────────────┐
│ ceremony_year │         title         │ is_winner │ movieId │ rating_year │ nb_ratings │ avg_rating │
│     int64     │        varchar        │  boolean  │  int64  │    int64    │   int64    │   double   │
├───────────────┼───────────────────────┼───────────┼─────────┼─────────────┼────────────┼────────────┤
│          1996 │ Apollo 13             │ false     │     150 │        1996 │      19102 │        4.0 │
│          1996 │ Apollo 13             │ false     │     150 │        1997 │       1624 │       3.98 │
│          1996 │ Apollo 13             │ false     │     150 │        1998 │        990 │       4.04 │
│          1996 │ Apollo 13             │ false     │     150 │        1999 │       2024 │        4.0 │
│          1996 │ Apollo 13             │ false     │     150 │        2000 │       2397 │       4.02 │
│          1996 │ Apollo 13             │ false     │     150 │ 

In [24]:
# Step 12: create a reusable view with the number of ratings and average rating per nominee and per rating year (UTC)
con.sql("CREATE VIEW film_year AS SELECT n.ceremony_year, n.title, n.is_winner, n.movieId, year(to_timestamp(r.timestamp) AT TIME ZONE 'UTC') AS rating_year, COUNT(*) AS nb_ratings, ROUND(AVG(r.rating), 2) AS avg_rating FROM '../data/processed/best_picture_nominees_1996_2017.csv' n JOIN '../data/raw/ml-32m/ratings.csv' r ON n.movieId = r.movieId GROUP BY n.ceremony_year, n.title, n.is_winner, n.movieId, rating_year")

In [25]:
# Step 15: how many film-year rows have fewer than N ratings, for several thresholds N (nothing is removed)
con.sql("SELECT COUNT(*) AS total_rows, SUM(CASE WHEN nb_ratings < 10 THEN 1 ELSE 0 END) AS below_10, SUM(CASE WHEN nb_ratings < 30 THEN 1 ELSE 0 END) AS below_30, SUM(CASE WHEN nb_ratings < 50 THEN 1 ELSE 0 END) AS below_50, SUM(CASE WHEN nb_ratings < 100 THEN 1 ELSE 0 END) AS below_100 FROM film_year")

┌────────────┬──────────┬──────────┬──────────┬───────────┐
│ total_rows │ below_10 │ below_30 │ below_50 │ below_100 │
│   int64    │  int128  │  int128  │  int128  │  int128   │
├────────────┼──────────┼──────────┼──────────┼───────────┤
│       2403 │       14 │       41 │      108 │       243 │
└────────────┴──────────┴──────────┴──────────┴───────────┘

In [26]:
# Step 16: same count, but only for the main analysis window (ceremony year + 1 to + 5)
con.sql("SELECT COUNT(*) AS total_rows, SUM(CASE WHEN nb_ratings < 10 THEN 1 ELSE 0 END) AS below_10, SUM(CASE WHEN nb_ratings < 30 THEN 1 ELSE 0 END) AS below_30, SUM(CASE WHEN nb_ratings < 50 THEN 1 ELSE 0 END) AS below_50, SUM(CASE WHEN nb_ratings < 100 THEN 1 ELSE 0 END) AS below_100 FROM film_year WHERE rating_year BETWEEN ceremony_year + 1 AND ceremony_year + 5")

┌────────────┬──────────┬──────────┬──────────┬───────────┐
│ total_rows │ below_10 │ below_30 │ below_50 │ below_100 │
│   int64    │  int128  │  int128  │  int128  │  int128   │
├────────────┼──────────┼──────────┼──────────┼───────────┤
│        710 │        0 │        2 │        6 │        28 │
└────────────┴──────────┴──────────┴──────────┴───────────┘

In [28]:
# Step 17: standard deviation of all ratings given to the 142 nominees
con.sql("SELECT ROUND(STDDEV(r.rating), 2) AS std_rating FROM '../data/processed/best_picture_nominees_1996_2017.csv' n JOIN '../data/raw/ml-32m/ratings.csv' r ON n.movieId = r.movieId")

┌────────────┐
│ std_rating │
│   double   │
├────────────┤
│       0.94 │
└────────────┘

In [29]:
# Step 19: view with only the main analysis window Y (ceremony year to ceremony year + 5), plus years since the ceremony
con.sql("CREATE VIEW analysis_window AS SELECT *, rating_year - ceremony_year AS years_since_ceremony FROM film_year WHERE rating_year BETWEEN ceremony_year AND ceremony_year + 5")

In [30]:
# check: the window has 6 years per film, 142 films each year since the ceremony
con.sql("SELECT years_since_ceremony, COUNT(*) AS n_rows FROM analysis_window GROUP BY years_since_ceremony ORDER BY years_since_ceremony")

┌──────────────────────┬────────┐
│ years_since_ceremony │ n_rows │
│        int64         │ int64  │
├──────────────────────┼────────┤
│                    0 │    142 │
│                    1 │    142 │
│                    2 │    142 │
│                    3 │    142 │
│                    4 │    142 │
│                    5 │    142 │
└──────────────────────┴────────┘